# 05 · YOLO Tuning (Kaggle)

**Goal:** compare YOLO11s tuning configurations on exactly the same GroupKFold fold 0, then validate the selected configuration on folds 1 and 2. The existing `01_EDA_YOLO_Baseline` notebook remains unchanged.

**Important evaluation distinction:** the baseline YOLO11n (3 folds, Ultralytics mAP) is valid as a within-framework baseline, but those numbers should **not** be directly ranked against torchvision models evaluated with COCOeval. This notebook adds a second, shared **COCOeval (AP@[.50:.95], 100 detections/image, original ignored/crowd regions)** protocol. For the final four-architecture comparison, recompute the *same* COCOeval for all checkpoints, on the *same* validation images.

Training budget: 6 epochs, batch=2, seed=42, frame_step=6, video-level GroupKFold. **Do not use original `val/` to tune**: it remains a final holdout.

**Kaggle:** Add competition data via **Add Input**, enable GPU, turn Internet on for installation/weight download if needed. Save the completed output artifacts externally or in a committed Kaggle notebook version; `/kaggle/working` in a Draft Session is temporary.

## 1 · Setup and dataset discovery

In [1]:
%pip -q install ultralytics pycocotools scikit-learn pyyaml

Note: you may need to restart the kernel to use updated packages.


In [2]:
from pathlib import Path
import os, json, time, gc, random, shutil
import numpy as np
import pandas as pd
import torch
import yaml
from sklearn.model_selection import GroupKFold
from ultralytics import YOLO
from pycocotools.coco import COCO
from pycocotools.cocoeval import COCOeval

SEED, FRAME_STEP, N_FOLDS = 42, 6, 3
BATCH, EPOCHS = 2, 6
CLASS_NAMES = ['person','bicycle','car','van','truck','tricycle','bus','motor']
DATA_ROOT = Path('/kaggle/input/competitions/multiple-object-video-detection-hw-2-ucu-ai-2026')
if not DATA_ROOT.exists():
    DATA_ROOT = Path('/kaggle/input/multiple-object-video-detection-hw-2-ucu-ai-2026')
assert (DATA_ROOT / 'train/annotations_coco.json').exists(), f'Attach competition dataset; not found at {DATA_ROOT}'
assert torch.cuda.is_available(), 'Enable a GPU accelerator in Kaggle settings.'
OUT = Path('/kaggle/working/yolo_tuning')
SPLITS = OUT/'splits'
RUNS = OUT/'runs'
EVAL = OUT/'coco_evaluation'
for p in (OUT,SPLITS,RUNS,EVAL): p.mkdir(parents=True,exist_ok=True)

random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
print('Dataset:', DATA_ROOT)
print('GPU:', torch.cuda.get_device_name(0))
print('Torch:', torch.__version__)
import ultralytics
print('Ultralytics:', ultralytics.__version__)


Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings
Dataset: /kaggle/input/competitions/multiple-object-video-detection-hw-2-ucu-ai-2026
GPU: Tesla T4
Torch: 2.11.0+cu130
Ultralytics: 8.4.204


## 2 · Reconstruct **exact same 3 folds** as original baseline

Uses chronological sampling `[::6]` per video and `GroupKFold(3)`, matching `01_EDA_YOLO_Baseline`. Persist all split IDs for reproducibility.

In [3]:
with open(DATA_ROOT/'train/annotations_coco.json') as f:
    train_coco_dict = json.load(f)
metadata = pd.DataFrame([{
    'image_id': int(im['id']),
    'file_name': im['file_name'],
    'clip': Path(im['file_name']).parent.name,
    'width': im['width'], 'height': im['height']
} for im in train_coco_dict['images']])
metadata = metadata.sort_values(['clip','file_name']).reset_index(drop=True)
# Equivalent to baseline groupby(...).apply(lambda group: group.iloc[::6]).
sampled = metadata.groupby('clip',group_keys=False).apply(lambda g: g.iloc[::FRAME_STEP], include_groups=False).reset_index(drop=True)
sampled['clip'] = sampled['file_name'].map(lambda x: Path(x).parent.name)
sampled['fold'] = -1
gkf = GroupKFold(n_splits=N_FOLDS)
for fold, (_, vi) in enumerate(gkf.split(sampled,groups=sampled['clip'])):
    sampled.loc[vi,'fold'] = fold
assert (sampled.fold>=0).all()
assert len(sampled)==4062, f'Expected 4062 sampled images; got {len(sampled)}. Check dataset/sampling.'
for fold in range(3):
    tr=sampled[sampled.fold!=fold]; va=sampled[sampled.fold==fold]
    assert set(tr['clip']).isdisjoint(set(va['clip']))
    print(f'fold={fold}: train={len(tr)}, val={len(va)}, train_clips={tr["clip"].nunique()}, val_clips={va["clip"].nunique()}')
sampled.to_csv(OUT/'cv_splits_step6.csv',index=False)


fold=0: train=2705, val=1357, train_clips=37, val_clips=19
fold=1: train=2706, val=1356, train_clips=37, val_clips=19
fold=2: train=2713, val=1349, train_clips=38, val_clips=18


## 3 · Build YOLO fold YAMLs (reuse original images and labels)

Check labels actually exist before training. This notebook deliberately does **not** copy the full multi-gigabyte dataset into `/kaggle/working`.

In [4]:
def train_image(file_name):
    return DATA_ROOT / 'train' / file_name

def label_from_image(p):
    return Path(str(p).replace('/images/','/labels/')).with_suffix('.txt')

FOLD_YAMLS={}
for fold in range(3):
    fd=SPLITS/f'fold_{fold}'; fd.mkdir(exist_ok=True,parents=True)
    for split,part in [('train',sampled[sampled.fold!=fold]),('val',sampled[sampled.fold==fold])]:
        paths=[train_image(fn) for fn in part['file_name']]
        missing_images=[str(p) for p in paths if not p.is_file()]
        missing_labels=[str(p) for p in paths if not label_from_image(p).is_file()]
        assert not missing_images, f'Missing images: {missing_images[:3]}'
        assert not missing_labels, f'Missing labels: {missing_labels[:3]}'
        (fd/f'{split}.txt').write_text('\n'.join(map(str,paths))+'\n')
    cfg={'train':str(fd/'train.txt'),'val':str(fd/'val.txt'),'nc':8,'names':dict(enumerate(CLASS_NAMES))}
    (fd/'data.yaml').write_text(yaml.safe_dump(cfg,sort_keys=False))
    FOLD_YAMLS[fold]=fd/'data.yaml'
print('All 3 folds ready:',FOLD_YAMLS)


All 3 folds ready: {0: PosixPath('/kaggle/working/yolo_tuning/splits/fold_0/data.yaml'), 1: PosixPath('/kaggle/working/yolo_tuning/splits/fold_1/data.yaml'), 2: PosixPath('/kaggle/working/yolo_tuning/splits/fold_2/data.yaml')}


## 4 · Tuning plan

The prior notebook already evaluated YOLO11s at **640 and 960 px on fold 0** with AdamW `lr0=0.001`. We do not silently reinterpret that result as a 3-fold baseline.

**New independent trials (fold 0 only)**: same YOLO11s, same 960 px, same six epochs; vary optimizer and/or LR.

| Trial | Optimizer | lr0 | Purpose |
|---|---|---:|---|
| Historical reference | AdamW | .001 | YOLO11s 960, trained in baseline notebook |
| `s960_adamw_lr0005` | AdamW | .0005 | Is the smaller LR better? |
| `s960_sgd_lr001` | SGD | .01 | Alternative optimizer configuration |
| `s960_adamw_lr002` | AdamW | .002 | Is the larger LR better? |

**Caution:** AdamW vs SGD trials use different learning rates, so their score differences are not a causal estimate of optimizer *alone*. For controlled optimizer comparisons, add matching learning-rate trials or do an explicit LR grid per optimizer.

Set `RUN_TRIALS` to run only the experiments you have GPU time for; default = first two. Runs skip completed training and retain checkpoints. The existing baseline files are never modified.

In [5]:
TRIALS = {
    's960_adamw_lr0005': {'model':'yolo11s.pt','imgsz':960,'optimizer':'AdamW','lr0':0.0005},
    's960_sgd_lr001': {'model':'yolo11s.pt','imgsz':960,'optimizer':'SGD','lr0':0.01},
    's960_adamw_lr002': {'model':'yolo11s.pt','imgsz':960,'optimizer':'AdamW','lr0':0.002},
}
RUN_TRIALS = ['s960_adamw_lr0005','s960_sgd_lr001']  # Add third trial if desired.
RUN_FOLD = 0
assert all(x in TRIALS for x in RUN_TRIALS)
print('Scheduled trials:', RUN_TRIALS)


Scheduled trials: ['s960_adamw_lr0005', 's960_sgd_lr001']


## 5 · Train experiment (one at a time) and resume safely

To prevent accidental retraining, a trial is skipped if its `training_complete.json` marker and `best.pt` exist. If interrupted, you can resume from `last.pt` by rerunning its cell; Ultralytics resume loads its original training settings. Confirm `results.csv` finished 6 epochs before relying on the marker.

In [6]:
def _train_dir(name,fold):
    return RUNS/f'{name}_fold_{fold}'

def train_trial(name, fold=0):
    assert name in TRIALS and fold in (0,1,2)
    cfg=TRIALS[name]
    name_fold=f'{name}_fold_{fold}'
    run_dir=_train_dir(name,fold)
    best=run_dir/'weights/best.pt'
    last=run_dir/'weights/last.pt'
    marker=run_dir/'training_complete.json'
    if marker.exists() and best.exists():
        print('Already trained, skipping:', name_fold)
        return best
    # Detect completed training that stopped before writing marker.
    history=run_dir/'results.csv'
    if history.exists() and best.exists():
        hist=pd.read_csv(history)
        if len(hist)>=EPOCHS:
            marker.write_text(json.dumps({'name':name,'fold':fold,'epochs':EPOCHS,'best':str(best)}))
            print('Training history complete; skipping:',name_fold)
            return best
    start=time.monotonic()
    if last.exists():
        print('Resuming from last checkpoint:',last)
        m=YOLO(str(last))
        m.train(resume=True, device=0, workers=2)
    else:
        m=YOLO(cfg['model'])
        m.train(
            data=str(FOLD_YAMLS[fold]), epochs=EPOCHS,
            imgsz=cfg['imgsz'],batch=BATCH,device=0,workers=2,
            optimizer=cfg['optimizer'],lr0=cfg['lr0'],seed=SEED,
            deterministic=True,amp=True,patience=EPOCHS,
            save=True,save_period=1,plots=False,cache=False,
            project=str(RUNS),name=name_fold,exist_ok=True,
            close_mosaic=10,
        )
    assert best.exists(), f'Expected best checkpoint at {best}'
    assert history.exists(), f'Expected results.csv at {history}'
    epochs_done=len(pd.read_csv(history))
    assert epochs_done>=EPOCHS, f'Training has {epochs_done}/{EPOCHS} epochs; do not mark complete.'
    marker.write_text(json.dumps({'name':name,'fold':fold,'epochs':epochs_done,
                                  'best':str(best),'train_minutes_this_call':(time.monotonic()-start)/60},indent=2))
    print('TRAINING COMPLETE:',name_fold,'best:',best)
    del m
    gc.collect();torch.cuda.empty_cache()
    return best


In [8]:
# Start with ONE trial. Each trial saves its own model weights and history.
# Uncomment to train the first tuning trial:
best_path = train_trial('s960_adamw_lr0005', fold=0)
print('Ready. Uncomment the train_trial call when ready.')


Ultralytics 8.4.204 🚀 Python-3.13.16 torch-2.11.0+cu130 CUDA:0 (Tesla T4, 14912MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=2, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/kaggle/working/yolo_tuning/splits/fold_0/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=6, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=960, iou=0.7, kobj=1.0, line_width=None, lr0=0.0005, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo11s.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=s960_adamw_lr0005_fold_0, nbs=64, nms=

/usr/local/lib/python3.13/dist-packages/ray/train/_internal/session.py:683: UserWarning: `get_trial_id` is meant to only be called inside a function that is executed by a Tuner or Trainer. Returning `None`.
  warnings.warn(


        2/6       1.5G      1.627      1.075       1.05         55        960: 100% ━━━━━━━━━━━━ 1353/1353 11.1it/s 2:020.2ss
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 340/340 15.0it/s 22.7s0.1ss
                   all       1357      59035       0.41      0.377      0.335      0.171

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
        3/6       1.5G      1.569     0.9794      1.029         81        960: 100% ━━━━━━━━━━━━ 1353/1353 11.1it/s 2:020.1ss
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 340/340 14.9it/s 22.8s0.1ss
                   all       1357      59035      0.451      0.404      0.354      0.181

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
        4/6       1.5G      1.523     0.9074      1.009         28        960: 100% ━━━━━━━━━━━━ 1353/1353 11.1it/s 2:020.2ss
              

In [9]:
# Run the rest sequentially only after confirming GPU/time budget.
# Can be rerun: completed trials are skipped.
for name in RUN_TRIALS:
    train_trial(name, fold=RUN_FOLD)
print('Uncomment the loop to run selected trials.')


Already trained, skipping: s960_adamw_lr0005_fold_0
Ultralytics 8.4.204 🚀 Python-3.13.16 torch-2.11.0+cu130 CUDA:0 (Tesla T4, 14912MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=2, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/kaggle/working/yolo_tuning/splits/fold_0/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=6, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=960, iou=0.7, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo11s.pt, momentum=0.937, mosaic=1.0, multi_scal

## 6 · Shared COCOeval (fold validation set, ignore/crowd-aware)

This is **not the same metric** as Ultralytics `model.val()`. It uses the original COCO annotation JSON, including `iscrowd`, and **COCOeval with maxDets = [1,10,100]**. The evaluation uses **the identical sampled validation image IDs** for the model under test.

Prediction conversion: YOLO class `0..7` → COCO category `1..8`; detections are mapped to **original pixel coordinates** and exported as COCO `[x,y,width,height]`. Use matching COCOeval settings for Faster R-CNN, RetinaNet and RT-DETR in the final comparison.

In [10]:
TRAIN_COCO_PATH = DATA_ROOT/'train/annotations_coco.json'
GT = COCO(str(TRAIN_COCO_PATH))
CATEGORY_IDS={int(c['id']) for c in GT.dataset['categories']}
assert CATEGORY_IDS==set(range(1,9)), f'Unexpected COCO category IDs: {CATEGORY_IDS}'

def coco_eval_yolo(checkpoint, fold=0, imgsz=960, tag=None, conf=0.001,
                   iou=0.7, max_det=300, predict_batch=8):
    checkpoint=Path(checkpoint)
    assert checkpoint.is_file(), f'Checkpoint not found: {checkpoint}'
    assert fold in (0,1,2)
    tag = tag or checkpoint.parent.parent.name
    outdir=EVAL/f'{tag}_fold_{fold}'
    outdir.mkdir(parents=True,exist_ok=True)
    summary_path=outdir/'coco_metrics.json'
    detections_path=outdir/'predictions_coco.json'
    if summary_path.exists() and detections_path.exists():
        print('COCOeval already completed:', summary_path)
        return json.loads(summary_path.read_text())

    val_table=sampled[sampled.fold==fold].sort_values('image_id')
    image_ids=val_table['image_id'].astype(int).tolist()
    image_files=[str(train_image(x)) for x in val_table['file_name']]
    assert len(image_ids)==len(image_files)>0
    model=YOLO(str(checkpoint))
    detections=[]
    start=time.monotonic()
    for i in range(0,len(image_files),predict_batch):
        paths=image_files[i:i+predict_batch]
        ids=image_ids[i:i+predict_batch]
        results=model.predict(source=paths, imgsz=imgsz, device=0, batch=predict_batch,
                              conf=conf,iou=iou,max_det=max_det,verbose=False,
                              augment=False,save=False)
        assert len(results)==len(ids), 'Prediction/image mapping mismatch.'
        for image_id,r in zip(ids,results):
            if r.boxes is None or len(r.boxes)==0: continue
            boxes=r.boxes.xyxy.cpu().numpy()
            classes=r.boxes.cls.int().cpu().tolist()
            scores=r.boxes.conf.cpu().tolist()
            for (x1,y1,x2,y2),cls,score in zip(boxes,classes,scores):
                if int(cls) not in range(8): continue
                w=max(0.,float(x2-x1)); h=max(0.,float(y2-y1))
                if w==0 or h==0: continue
                detections.append({'image_id':int(image_id),'category_id':int(cls)+1,
                                   'bbox':[float(x1),float(y1),w,h],
                                   'score':float(score)})
        if (i//predict_batch+1)%25==0:
            print(f'Inference: {min(i+predict_batch,len(image_files))}/{len(image_files)} images')
    inference_seconds=time.monotonic()-start
    detections_path.write_text(json.dumps(detections))
    assert detections, 'No predictions collected. Check model path and confidence threshold.'
    dt=GT.loadRes(detections)
    evaluator=COCOeval(GT,dt,iouType='bbox')
    evaluator.params.imgIds=image_ids
    evaluator.params.catIds=list(range(1,9))
    evaluator.params.maxDets=[1,10,100]
    evaluator.evaluate(); evaluator.accumulate(); evaluator.summarize()
    s=evaluator.stats
    metrics={
        'tag':tag,'fold':fold,'checkpoint':str(checkpoint),'imgsz':imgsz,
        'n_val_images':len(image_ids),'n_predictions':len(detections),
        'mAP50-95':float(s[0]), 'mAP50':float(s[1]),'mAP75':float(s[2]),
        'AP_small':float(s[3]),'AP_medium':float(s[4]),'AP_large':float(s[5]),
        'inference_ms_per_image':round(inference_seconds/len(image_ids)*1000,2),
        'evaluator':'pycocotools COCOeval bbox, original iscrowd, maxDets=100'
    }
    summary_path.write_text(json.dumps(metrics,indent=2))
    del model,dt,evaluator
    gc.collect();torch.cuda.empty_cache()
    print('Saved:', summary_path)
    return metrics


loading annotations into memory...
Done (t=5.90s)
creating index...
index created!


## 7 · Evaluate tuning trial(s)

Run this after training. It will not retrain a model. Each trial writes `coco_metrics.json` and `predictions_coco.json`; predictions can be reused in the final comparison without GPU.

In [11]:
# Run COCOeval for any completed fold-0 tuning trials.
# This cell is safe to rerun; completed evaluation is skipped.
for name in RUN_TRIALS:
    ckpt=_train_dir(name,0)/'weights/best.pt'
    if ckpt.exists() and (_train_dir(name,0)/'training_complete.json').exists():
        result=coco_eval_yolo(ckpt, fold=0, imgsz=TRIALS[name]['imgsz'],tag=name)
        print(name,'COCO mAP50-95:',result['mAP50-95'])
    else:
        print('Not trained yet:',name)


Inference: 200/1357 images
Inference: 400/1357 images
Inference: 600/1357 images
Inference: 800/1357 images
Inference: 1000/1357 images
Inference: 1200/1357 images
Loading and preparing results...
DONE (t=0.43s)
creating index...
index created!
Running per image evaluation...
Evaluate annotation type *bbox*
DONE (t=89.72s).
Accumulating evaluation results...
DONE (t=3.43s).
 Average Precision  (AP) @[ IoU=0.50:0.95 | area=   all | maxDets=100 ] = 0.168
 Average Precision  (AP) @[ IoU=0.50      | area=   all | maxDets=100 ] = 0.323
 Average Precision  (AP) @[ IoU=0.75      | area=   all | maxDets=100 ] = 0.152
 Average Precision  (AP) @[ IoU=0.50:0.95 | area= small | maxDets=100 ] = 0.057
 Average Precision  (AP) @[ IoU=0.50:0.95 | area=medium | maxDets=100 ] = 0.151
 Average Precision  (AP) @[ IoU=0.50:0.95 | area= large | maxDets=100 ] = 0.334
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=   all | maxDets=  1 ] = 0.091
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=   all | maxDe

## 8 · Evaluate historical baseline checkpoint **without retraining**

**Critical for fair model comparisons:** the original YOLO11n and YOLO11s models are not magically available in this fresh Kaggle notebook. Attach a **saved output dataset** of the baseline notebook (or upload checkpoints as an input dataset), then set the path(s) below.

For the **YOLO11n 3-fold baseline**, evaluate its `best.pt` from each fold with **imgsz=640**, using `coco_eval_yolo(... fold=fold)`. For the historical **YOLO11s 960 fold 0** checkpoint, use **imgsz=960**. This is inference only, no new training. Otherwise, mark these metrics as *missing*, not comparable.

Kaggle may display previously saved notebook outputs under `/kaggle/input/<your-output-dataset>/...`; paths vary. The old baseline's original runtime `/kaggle/working/` path does NOT carry over.

In [13]:
# Optional: inspect available attached checkpoints (may take a few seconds).
# Limit discovery to attached Kaggle inputs rather than the whole filesystem.
found=[]
for p in Path('/kaggle/input').rglob('best.pt'):
    found.append(str(p))
print('Attached best.pt checkpoints:',len(found))
for p in found[:35]: print(p)


Attached best.pt checkpoints: 5
/kaggle/input/notebooks/margopaduchak/01-eda-yolo-baseline/yolo_experiments/yolo11s_960_adamw/weights/best.pt
/kaggle/input/notebooks/margopaduchak/01-eda-yolo-baseline/yolo_experiments/yolo11s_640_adamw/weights/best.pt
/kaggle/input/notebooks/margopaduchak/01-eda-yolo-baseline/yolo_baseline/fold_2/weights/best.pt
/kaggle/input/notebooks/margopaduchak/01-eda-yolo-baseline/yolo_baseline/fold_1/weights/best.pt
/kaggle/input/notebooks/margopaduchak/01-eda-yolo-baseline/yolo_baseline/fold_0/weights/best.pt


In [14]:

from pathlib import Path

BASE = Path(
    "/kaggle/input/notebooks/margopaduchak/"
    "01-eda-yolo-baseline"
)

YOLO11N_BASELINE_WEIGHTS = {
    0: str(BASE / "yolo_baseline/fold_0/weights/best.pt"),
    1: str(BASE / "yolo_baseline/fold_1/weights/best.pt"),
    2: str(BASE / "yolo_baseline/fold_2/weights/best.pt"),
}

YOLO11S_640_FOLD0_WEIGHTS = str(
    BASE / "yolo_experiments/yolo11s_640_adamw/weights/best.pt"
)

YOLO11S_960_FOLD0_WEIGHTS = str(
    BASE / "yolo_experiments/yolo11s_960_adamw/weights/best.pt"
)

# Verify all checkpoints
for path in [
    *YOLO11N_BASELINE_WEIGHTS.values(),
    YOLO11S_640_FOLD0_WEIGHTS,
    YOLO11S_960_FOLD0_WEIGHTS,
]:
    assert Path(path).is_file(), f"Missing: {path}"

print("All 5 historical checkpoints found.")

# Evaluate YOLO11n on all three folds
for fold, checkpoint in YOLO11N_BASELINE_WEIGHTS.items():
    coco_eval_yolo(
        checkpoint,
        fold=fold,
        imgsz=640,
        tag=f"baseline_yolo11n_fold{fold}"
    )

# Evaluate YOLO11s at 640 and 960 on fold 0
coco_eval_yolo(
    YOLO11S_640_FOLD0_WEIGHTS,
    fold=0,
    imgsz=640,
    tag="historical_yolo11s_640_adamw"
)

coco_eval_yolo(
    YOLO11S_960_FOLD0_WEIGHTS,
    fold=0,
    imgsz=960,
    tag="historical_yolo11s_960_adamw"
)

print("Historical baseline COCO evaluation completed.")


All 5 historical checkpoints found.
Inference: 200/1357 images
Inference: 400/1357 images
Inference: 600/1357 images
Inference: 800/1357 images
Inference: 1000/1357 images
Inference: 1200/1357 images
Loading and preparing results...
DONE (t=0.46s)
creating index...
index created!
Running per image evaluation...
Evaluate annotation type *bbox*
DONE (t=91.40s).
Accumulating evaluation results...
DONE (t=3.22s).
 Average Precision  (AP) @[ IoU=0.50:0.95 | area=   all | maxDets=100 ] = 0.105
 Average Precision  (AP) @[ IoU=0.50      | area=   all | maxDets=100 ] = 0.205
 Average Precision  (AP) @[ IoU=0.75      | area=   all | maxDets=100 ] = 0.095
 Average Precision  (AP) @[ IoU=0.50:0.95 | area= small | maxDets=100 ] = 0.023
 Average Precision  (AP) @[ IoU=0.50:0.95 | area=medium | maxDets=100 ] = 0.089
 Average Precision  (AP) @[ IoU=0.50:0.95 | area= large | maxDets=100 ] = 0.207
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=   all | maxDets=  1 ] = 0.056
 Average Recall     (AR) @[

## 9 · Compare finished COCO evaluation results

**Comparison guardrails:** rank fold 0 models only against other **fold 0** models, evaluated on the same image IDs. Do not mix fold 0 results with three-fold means. Do not mix Ultralytics mAP with COCOeval mAP.

In [18]:
rows=[]
for p in EVAL.glob('*/coco_metrics.json'):
    rows.append(json.loads(p.read_text()))
if not rows:
    print('No COCOeval results yet; train + evaluate or attach baseline checkpoints.')
else:
    all_eval=pd.DataFrame(rows)
    all_eval.to_csv(OUT/'coco_comparison_all.csv',index=False)
    fold0=all_eval[all_eval['fold']==0].sort_values('mAP50-95',ascending=False)
    display(fold0[['tag','fold','imgsz','mAP50-95','mAP50','AP_small','inference_ms_per_image']])
    print('Saved:',OUT/'coco_comparison_all.csv')


,tag,fold,imgsz,mAP50-95,mAP50,AP_small,inference_ms_per_image
0,s960_adamw_lr0005,0,960,0.168237,0.323392,0.056723,43.48
4,historical_yolo11s_960_adamw,0,960,0.167859,0.320178,0.054223,42.05
7,s960_sgd_lr001,0,960,0.163198,0.314546,0.060122,41.03
6,historical_yolo11s_640_adamw,0,640,0.130679,0.254778,0.035336,34.71
1,baseline_yolo11n_fold0,0,640,0.104531,0.204839,0.023091,29.60


Saved: /kaggle/working/yolo_tuning/coco_comparison_all.csv


## 10 · Optional: validate selected tuned configuration on folds 1 & 2

**Do this only after choosing a configuration from fold-0 results.** A 3-fold tuning-validation mean is helpful, but after selecting hyperparameters using fold 0 it is not an unbiased nested-CV performance estimate. The untouched original validation holdout is reserved for final assessment.

All models start from pretrained `yolo11s.pt` and train on the appropriate fold-specific training images. Do **not** fine-tune fold 1 from a model already trained on fold 0 (that would leak fold 1 validation videos).

In [16]:
# Example (uncomment AFTER reviewing the fold-0 leaderboard):
SELECTED_TRIAL = 's960_adamw_lr0005'
for fold in [1, 2]:
    checkpoint = train_trial(SELECTED_TRIAL,fold=fold)
    coco_eval_yolo(checkpoint,fold=fold,imgsz=TRIALS[SELECTED_TRIAL]['imgsz'],tag=SELECTED_TRIAL)
print('Folds 1 and 2 are deliberately opt-in.')


Ultralytics 8.4.204 🚀 Python-3.13.16 torch-2.11.0+cu130 CUDA:0 (Tesla T4, 14912MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=2, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/kaggle/working/yolo_tuning/splits/fold_1/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=6, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=960, iou=0.7, kobj=1.0, line_width=None, lr0=0.0005, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo11s.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=s960_adamw_lr0005_fold_1, nbs=64, nms=

## 11 · Export results / backup essential files

Creates a compact ZIP of CSV, JSON, YAML, and `best.pt` checkpoints (excluding redundant per-epoch `epoch*.pt`, `last.pt`, images and large caches). **Download from Kaggle Output panel** or save as a committed version. The ZIP still lives in `/kaggle/working` until exported; it is not a permanent backup by itself. To resume interrupted training, keep `last.pt` separately.

In [22]:
from zipfile import ZipFile, ZIP_STORED
ARCHIVE=Path('/kaggle/working/yolo_tuning_results.zip')
include=[]
for p in OUT.rglob('*'):
    if not p.is_file(): continue
    if p.suffix.lower() in ('.csv','.json','.yaml','.txt'):
        # Skip duplicate YOLO predictions JSON if archive becomes too large? Retain these for comparisons.
        include.append(p)
    elif p.name=='best.pt':
        include.append(p)
with ZipFile(ARCHIVE,'w',ZIP_STORED) as z:
    for p in include: z.write(p,arcname=str(p.relative_to(OUT)))
with ZipFile(ARCHIVE) as z:
    assert z.testzip() is None
print('Archive:',ARCHIVE)
print('Size MB:',round(ARCHIVE.stat().st_size/1e6,1))
print('Files:',len(include))
print('Download via Kaggle Output browser / saved notebook version, not IPython FileLink.')


Archive: /kaggle/working/yolo_tuning_results.zip
Size MB: 552.3
Files: 47
Download via Kaggle Output browser / saved notebook version, not IPython FileLink.


In [21]:
import json
import pandas as pd
from pathlib import Path

eval_dir = Path("/kaggle/working/yolo_tuning/coco_evaluation")

records = []
for fold in (0, 1, 2):
    path = (
        eval_dir
        / f"s960_adamw_lr0005_fold_{fold}"
        / "coco_metrics.json"
    )
    assert path.exists(), f"Missing metrics for fold {fold}"
    records.append(json.loads(path.read_text()))

df = pd.DataFrame(records).sort_values("fold")

columns = [
    "fold", "mAP50-95", "mAP50", "mAP75",
    "AP_small", "AP_medium", "AP_large"
]

display(df[columns])

summary = df[columns[1:]].agg(["mean", "std"])
display(summary)

summary.to_csv(
    "/kaggle/working/yolo_tuning/yolo_tuned_3fold_summary.csv"
)
df.to_csv(
    "/kaggle/working/yolo_tuning/yolo_tuned_3fold_results.csv",
    index=False
)

print("3-fold results and summary saved.")

,fold,mAP50-95,mAP50,mAP75,AP_small,AP_medium,AP_large
0,0,0.168237,0.323392,0.151950,0.056723,0.150665,0.334298
1,1,0.183649,0.334303,0.177620,0.039360,0.169799,0.420751
2,2,0.171291,0.331779,0.158515,0.094309,0.180712,0.291843


,mAP50-95,mAP50,mAP75,AP_small,AP_medium,AP_large
mean,0.174392,0.329825,0.162695,0.063464,0.167059,0.348964
std,0.008160,0.005712,0.013336,0.028088,0.015210,0.065694


3-fold results and summary saved.
